In [1]:
# Inicializar bibliotecas
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10

print("✅ Bibliotecas carregadas!")

✅ Bibliotecas carregadas!


In [ ]:
# Carregar dados com delimitador correto
caminho_arquivo = r'D:\OneDrive\Pessoais\Doutorado\Cefet\data\bd_gta_dentro_mg202505091607.csv'

print("Carregando dados com delimitador ';'...")
# Ler apenas algumas linhas primeiro para inspecionar
df = pd.read_csv(caminho_arquivo, sep=';', nrows=100000, low_memory=False)

print(f"\n✅ Dados carregados: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\nColunas: {list(df.columns)}")
print(f"\nPrimeiras linhas:")
print(df.head())

In [ ]:
# Estatísticas descritivas
print("\n📊 ANÁLISE EXPLORATÓRIA")
print("="*70)
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores ausentes:")
print(df.isnull().sum())
print("\nEstatísticas das colunas numéricas:")
print(df.describe())

In [ ]:
# Processar coluna 'qtd' (quantidade)
# Converter para numérico, removendo valores inválidos
df['qtd'] = pd.to_numeric(df['qtd'], errors='coerce')

# Remover NaN
populacao = df['qtd'].dropna().values

# Análise da distribuição
mu_pop = np.mean(populacao)
sigma_pop = np.std(populacao, ddof=0)
mediana_pop = np.median(populacao)
assimetria_pop = stats.skew(populacao)
curtose_pop = stats.kurtosis(populacao)

print("\n📌 CARACTERIZAÇÃO DA VARIÁVEL 'QTD' (Quantidade)")
print("="*70)
print(f"Observações válidas: {len(populacao):,}")
print(f"Média: {mu_pop:.4f}")
print(f"Mediana: {mediana_pop:.4f}")
print(f"Desvio padrão: {sigma_pop:.4f}")
print(f"Assimetria: {assimetria_pop:.4f}", end="")

if abs(assimetria_pop) < 0.5:
    print(" → Aproximadamente simétrica")
elif assimetria_pop > 0.5:
    print(" → Assimétrica à direita (cauda positiva)")
else:
    print(" → Assimétrica à esquerda (cauda negativa)")

print(f"Curtose: {curtose_pop:.4f}")
print(f"Mínimo: {populacao.min():.4f}")
print(f"Máximo: {populacao.max():.4f}")
print(f"Amplitude: {populacao.max() - populacao.min():.4f}")

In [ ]:
# Visualizar distribuição
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(populacao, bins=100, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(mu_pop, color='red', linestyle='--', linewidth=2, label=f'Média = {mu_pop:.2f}')
axes[0].axvline(mediana_pop, color='green', linestyle='--', linewidth=2, label=f'Mediana = {mediana_pop:.2f}')
axes[0].set_xlabel('Quantidade (qtd)')
axes[0].set_ylabel('Frequência')
axes[0].set_title(f'Distribuição da Quantidade - Assimetria = {assimetria_pop:.2f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Q-Q Plot
stats.probplot(populacao, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot: Quantidade vs Distribuição Normal')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Distribuição visualizada!")

In [ ]:
# TEOREMA CENTRAL DO LIMITE (TCL)
print("\n📌 TEOREMA CENTRAL DO LIMITE")
print("="*70)

# Diferentes tamanhos de amostra
tamanhos_amostra = [5, 10, 30, 100]
num_amostras = 5000

print(f"Número de amostras para cada tamanho: {num_amostras:,}")

resultados_tcl = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n in enumerate(tamanhos_amostra):
    # Gerar amostras e calcular suas médias
    medias = np.array([np.mean(np.random.choice(populacao, size=n, replace=True)) 
                      for _ in range(num_amostras)])
    resultados_tcl[n] = medias
    
    # Estatísticas
    media_das_medias = np.mean(medias)
    dp_das_medias = np.std(medias, ddof=1)
    teorico_dp_das_medias = sigma_pop / np.sqrt(n)
    assimetria_medias = stats.skew(medias)
    
    print(f"\nn = {n}:")
    print(f"  Média das médias: {media_das_medias:.4f} (esperado: {mu_pop:.4f})")
    print(f"  DP das médias: {dp_das_medias:.4f} (teórico: {teorico_dp_das_medias:.4f})")
    print(f"  Assimetria: {assimetria_medias:.4f} (diminui conforme n aumenta)")
    
    # Visualização
    ax = axes[idx]
    ax.hist(medias, bins=50, color='skyblue', alpha=0.7, edgecolor='black', density=True)
    
    # Distribuição normal teórica
    x = np.linspace(medias.min(), medias.max(), 100)
    y_normal = stats.norm.pdf(x, mu_pop, teorico_dp_das_medias)
    ax.plot(x, y_normal, 'r-', linewidth=2, label='Normal teórica')
    
    ax.axvline(media_das_medias, color='green', linestyle='--', linewidth=2, label='Média observada')
    ax.set_xlabel('Média da Amostra')
    ax.set_ylabel('Densidade')
    ax.set_title(f'TCL: n = {n}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ TCL verificado: As médias convergem para distribuição normal!")

In [ ]:
# INTERVALO DE CONFIANÇA (IC)
print("\n📌 INTERVALO DE CONFIANÇA")
print("="*70)

nivel_confianca = 0.95
alpha = 1 - nivel_confianca
tamanho_amostra_ic = 100
num_amostras_ic = 1000

# Gerar amostras
amostras_ic = [np.random.choice(populacao, size=tamanho_amostra_ic, replace=True) 
               for _ in range(num_amostras_ic)]

# Calcular ICs
ics = []
contem_media = 0

for amostra in amostras_ic:
    x_bar = np.mean(amostra)
    s = np.std(amostra, ddof=1)
    se = s / np.sqrt(tamanho_amostra_ic)
    t_critico = stats.t.ppf(1 - alpha/2, df=tamanho_amostra_ic - 1)
    margem_erro = t_critico * se
    ic_inf = x_bar - margem_erro
    ic_sup = x_bar + margem_erro
    ics.append((ic_inf, ic_sup))
    
    if ic_inf <= mu_pop <= ic_sup:
        contem_media += 1

print(f"Tamanho de amostra: n = {tamanho_amostra_ic}")
print(f"Número de amostras: {num_amostras_ic}")
print(f"Nível de confiança: {nivel_confianca*100:.0f}%")
print(f"\nICs que contêm a média: {contem_media}/{num_amostras_ic} ({contem_media/num_amostras_ic*100:.1f}%)")
print(f"Esperado: ~{nivel_confianca*100:.0f}%")

# Visualizar
num_vis = 100
fig, ax = plt.subplots(figsize=(12, 8))

for i in range(num_vis):
    ic_inf, ic_sup = ics[i]
    cor = 'green' if ic_inf <= mu_pop <= ic_sup else 'red'
    ax.plot([ic_inf, ic_sup], [i, i], color=cor, alpha=0.6, linewidth=1)
    ax.plot(np.mean([ic_inf, ic_sup]), i, 'o', color=cor, markersize=3)

ax.axvline(mu_pop, color='blue', linestyle='--', linewidth=2, label=f'Média = {mu_pop:.2f}')
ax.set_xlabel('Valor')
ax.set_ylabel(f'Amostra (primeiras {num_vis})')
ax.set_title(f'Intervalos de Confiança {nivel_confianca*100:.0f}% para μ (n={tamanho_amostra_ic})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ ICs calculados!")

In [ ]:
# ANOVA - Comparar quantidades por estado de origem
print("\n📌 ANOVA - ANÁLISE DE VARIÂNCIA")
print("="*70)

# Verificar colunas disponíveis para estratificação
print("\nColunas categóricas disponíveis:")
colunas_cat = df.select_dtypes(include=['object']).columns.tolist()
print(colunas_cat)

# Usar 'estado_origem' para criar grupos
if 'estado_origem' in df.columns:
    print("\n🔍 Analisando quantidade por estado de origem...")
    
    # Criar grupos por estado
    grupos = []
    labels_grupos = []
    
    for estado in df['estado_origem'].unique()[:5]:  # Limitar a 5 estados para análise
        dados_grupo = df[df['estado_origem'] == estado]['qtd'].dropna().values
        if len(dados_grupo) > 30:  # Mínimo de observações
            grupos.append(dados_grupo)
            labels_grupos.append(str(estado))
    
    if len(grupos) >= 2:
        print(f"\nGrupos criados: {len(grupos)} estados")
        print(f"\nEstatísticas dos grupos:")
        
        for label, grupo in zip(labels_grupos, grupos):
            print(f"\n{label}: n = {len(grupo):,}")
            print(f"  Média: {np.mean(grupo):.4f}")
            print(f"  DP: {np.std(grupo, ddof=1):.4f}")
        
        # ANOVA
        f_stat, p_value = stats.f_oneway(*grupos)
        
        print(f"\n{'RESULTADO DA ANOVA':^70}")
        print(f"Estatística F: {f_stat:.4f}")
        print(f"p-value: {p_value:.10f}")
        
        if p_value < 0.05:
            print("✅ SIGNIFICATIVO: Há diferenças entre os estados")
        else:
            print("❌ NÃO SIGNIFICATIVO: Sem diferenças entre os estados")
        
        # Visualizar
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Boxplot
        bp = axes[0].boxplot(grupos, labels=labels_grupos, patch_artist=True)
        cores = plt.cm.Set3(np.linspace(0, 1, len(grupos)))
        for patch, color in zip(bp['boxes'], cores):
            patch.set_facecolor(color)
        axes[0].set_ylabel('Quantidade')
        axes[0].set_title('Boxplot por Estado')
        axes[0].grid(alpha=0.3, axis='y')
        
        # Histogramas
        for grupo, label, color in zip(grupos, labels_grupos, cores):
            axes[1].hist(grupo, bins=30, alpha=0.5, label=label, color=color, edgecolor='black')
        axes[1].set_xlabel('Quantidade')
        axes[1].set_ylabel('Frequência')
        axes[1].set_title('Distribuição por Estado')
        axes[1].legend(fontsize=8)
        axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("\n✅ ANOVA realizada!")
    else:
        print("\n⚠️  Dados insuficientes para ANOVA")
else:
    print("\n⚠️  Coluna 'estado_origem' não encontrada")

In [ ]:
# RESUMO FINAL
print("\n" + "="*70)
print("📊 RESUMO DA ANÁLISE COM DADOS REAIS")
print("="*70)

print(f"\n✅ Análises realizadas:")
print(f"   1. Análise Exploratória (n = {len(populacao):,})")
print(f"   2. Verificação de Distribuição (Assimetria = {assimetria_pop:.2f})")
print(f"   3. Teorema Central do Limite")
print(f"   4. Intervalo de Confiança (95%)")
if len(grupos) >= 2:
    print(f"   5. ANOVA (F = {f_stat:.2f}, p < 0.05)")

print(f"\n📈 Características dos Dados:")
print(f"   • Média: {mu_pop:.2f}")
print(f"   • Desvio padrão: {sigma_pop:.2f}")
print(f"   • Assimetria: {assimetria_pop:.2f}", end="")
if abs(assimetria_pop) < 0.5:
    print(" (Simétrica)")
elif assimetria_pop > 0:
    print(" (Assimétrica à direita)")
else:
    print(" (Assimétrica à esquerda)")

print(f"\n✅ Análise concluída com sucesso!")
print("\n" + "="*70)